In [ ]:
import sys
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import polars as pl
import numpy as np

project_root = Path.cwd()
while not (project_root / "requirements.txt").exists() and project_root != project_root.parent:
    project_root = project_root.parent
sys.path.append(str(project_root))
from scipy.stats import norm
from tests.fixtures.synthetic_data import synthetic_actual_df, synthetic_previous_df

from pricing.fair_value import (
get_daily_bucket,
)

from data.fetcher import (
fetch_polymarket_data,
fetch_polymarket_price_history,
fetch_all_price_history,
build_polymarket_price_dataset,

)
from data.fetcher import fetch_data,call_fetcher_functions
from data.cleaner import clean_data
from data.loader import add_event_column,filter_summer,get_separate_summer_months

from data.cleaner import clean_data,clean_polymarket_data
from data.loader import (
add_event_column,
filter_summer,
add_market_prob_column,
filter_resolved,
)
from pricing.edge import (
prob_market_v_model,

)
from models.baseline import (
gaussian_probability,
)
from models.kde_model import (
kde_estimate,
)
from models.bayesian_model import ( 
posterior_probability,
)
from pricing.fair_value import (
build_probability_vector,
build_daily_probability_vector,
)
from config.settings import (
HISTORICAL_START,
HISTORICAL_END,
DEFAULT_CITY,
TOMMORROWS_DATE,
FORECAST_START,
FORECAST_END,
LOWER_BOUND,
UPPER_BOUND,
IS_START,
IS_END,
OOS_START,
OOS_END,
)

In [ ]:


import sys
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import polars as pl
import numpy as np

project_root = Path.cwd()
while not (project_root / "requirements.txt").exists() and project_root != project_root.parent:
    project_root = project_root.parent
sys.path.append(str(project_root))
from scipy.stats import norm
from tests.fixtures.synthetic_data import synthetic_actual_df, synthetic_previous_df

from pricing.fair_value import (
get_daily_bucket,
)

from data.fetcher import (
fetch_polymarket_data,
fetch_polymarket_price_history,
fetch_all_price_history,
build_polymarket_price_dataset,

)
from data.fetcher import fetch_data,call_fetcher_functions
from data.cleaner import clean_data
from data.loader import add_event_column,filter_summer,get_separate_summer_months

from data.cleaner import clean_data,clean_polymarket_data
from data.loader import (
add_event_column,
filter_summer,
add_market_prob_column,
filter_resolved,
)
from pricing.edge import (
prob_market_v_model,
effective_edge,
)
from models.baseline import (
gaussian_probability,
)
from models.kde_model import (
kde_estimate,
)
from models.bayesian_model import ( 
posterior_probability,
)
from pricing.fair_value import (
build_probability_vector,
build_daily_probability_vector,
)

from backtest.engine import (
engine,
)
from config.settings import (
HISTORICAL_START,
HISTORICAL_END,
DEFAULT_CITY,
TOMMORROWS_DATE,
FORECAST_START,
FORECAST_END,
LOWER_BOUND,
UPPER_BOUND,
IS_START,
IS_END,
OOS_START,
OOS_END,
)


df_raw = fetch_data(IS_START,IS_END)
df_clean = clean_data(df_raw)
df_event = add_event_column(df_clean)
df_temp_summer = filter_summer(df_event)
df_summer = df_temp_summer["temperature_2m_max"].to_list()

df_pair = call_fetcher_functions(OOS_START, OOS_END)

df_result = build_polymarket_price_dataset()
print(df_result)
lst_buckets = get_daily_bucket(df_result)
lst_days  = df_result["date"].to_list()
#print(lst_buckets)

gaussian_prob_fn = lambda low, high: gaussian_probability(df_summer, low, high)[0]
kde_prob_fn = lambda low, high: kde_estimate(df_summer, low, high)
bayesian_prob_fn = lambda day, low, high: posterior_probability(df_summer, day, df_pair, low, high)

p_gauss = build_probability_vector(gaussian_prob_fn, lst_buckets)
p_kde =  build_probability_vector(kde_prob_fn, lst_buckets)
p_bayes = build_daily_probability_vector(bayesian_prob_fn, lst_buckets,lst_days)

edge_gauss = prob_market_v_model(p_gauss,df_result)
edge_kde = prob_market_v_model(p_kde ,df_result)
edge_bayes = prob_market_v_model(p_bayes ,df_result)

#gauss_effective = effective_edge(edge_gauss)
#kde_effective = effective_edge(edge_kde)
#bayes_effective = effective_edge(edge_bayes)




In [ ]:
# 1. Skapa en figur med 1 rad och 3 kolumner
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 2. Rita första grafen på axes[0]
sns.histplot(data=edge_gauss["edge"].to_frame(), x ="edge", ax=axes[0],binwidth=0.03)
axes[0].set_title("Gauss")

# 3. Rita andra grafen på axes[1]
sns.histplot(data=edge_kde["edge"].to_frame(), x="edge", ax=axes[1],binwidth=0.03)
axes[1].set_title("KDE")

# 4. Rita tredje grafen på axes[2]
sns.histplot(data=edge_bayes["edge"].to_frame(), x="edge", ax=axes[2],binwidth=0.03)
axes[2].set_title("Bayes")

# Justera avståndet så att rubriker inte överlappar
plt.tight_layout()
plt.show()

Plotted the edge over time, we can see that Gauss and KDE are very similiar, almost hard to spot any sudden difference. We can see that all three graphs has maximun at 0 edge, which means the markets prediction vs wheaters predion are almost the same. We can spot a shift in that Bayes have more bins at the postive edge the  Gauss and KDE do.

In [ ]:

import sys
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import polars as pl
import numpy as np

project_root = Path.cwd()
while not (project_root / "requirements.txt").exists() and project_root != project_root.parent:
    project_root = project_root.parent
sys.path.append(str(project_root))
from scipy.stats import norm
from evaluation.eval_loop import run_eval_loop_polymarket, make_static_factory, bayes_static_factory
from pricing.fair_value import create_buckets
from config.settings import (
HISTORICAL_START,
HISTORICAL_END,
DEFAULT_CITY,
TOMMORROWS_DATE,
FORECAST_START,
FORECAST_END,
LOWER_BOUND,
UPPER_BOUND,
IS_START,
IS_END,
OOS_START,
OOS_END,
)


gauss_factory = make_static_factory(gaussian_prob_fn)
kde_factory = make_static_factory(kde_prob_fn)
bayes_factory =  bayes_static_factory(bayesian_prob_fn)

buckets = create_buckets(LOWER_BOUND,UPPER_BOUND) 

df_res_gauss = run_eval_loop_polymarket(gauss_factory,buckets,df_pair,df_result)
df_res_kde = run_eval_loop_polymarket(kde_factory,buckets,df_pair,df_result)
df_res_bayes = run_eval_loop_polymarket(bayes_factory,buckets,df_pair,df_result)


gauss_engine = engine(p_gauss,df_res_gauss)
kde_engine = engine(p_kde,df_res_kde)

bayes_engine = engine(p_bayes,df_res_bayes)

print(gauss_engine)

print(kde_engine)

print(bayes_engine)





In [ ]:
import sys
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import polars as pl
import numpy as np

project_root = Path.cwd()
while not (project_root / "requirements.txt").exists() and project_root != project_root.parent:
    project_root = project_root.parent
sys.path.append(str(project_root))

from data.fetcher import build_polymarket_price_dataset,get_spread_polymarket
from risk.metrics import value_at_risk,expected_shortfall,rolling_max_dd
#df_result = build_polymarket_price_dataset()
#spread = get_spread_polymarket()
#print(spread)


vr_gauss = value_at_risk(gauss_engine)
vr_kde = value_at_risk(kde_engine)
vr_bayes = value_at_risk(bayes_engine)

es_gauss = expected_shortfall(gauss_engine,vr_gauss)
es_kde = expected_shortfall(kde_engine,vr_kde)
es_bayes = expected_shortfall(bayes_engine,vr_bayes)


rm_gauss = rolling_max_dd(gauss_engine)
rm_kde = rolling_max_dd(kde_engine)
rm_bayes = rolling_max_dd(bayes_engine)

print(vr_gauss)
print(vr_kde)
print(vr_bayes)

print(es_gauss)
print(es_kde)
print(es_bayes)

print(rm_gauss)
print(rm_kde)
print(rm_bayes)



